In [ ]:
from collections.abc import Iterable

import matplotlib.pyplot as plt
import numpy as np
import scipp as sc
from ipywidgets import widgets, Layout
# from IPython.display import HTML, display

from vector_alignment import Goniometer, GoniometerAxis

In [ ]:
%matplotlib widget

In [ ]:
axes = [
    GoniometerAxis(
        name="gcu",
        axis=sc.vector([1, 0, 0]),
        min=sc.scalar(-30, unit="degree"),
        max=sc.scalar(30, unit="degree"),
    ),
    GoniometerAxis(
        name="gcl",
        axis=sc.vector([0, 0, 1]),
        min=sc.scalar(-30, unit="degree"),
        max=sc.scalar(30, unit="degree"),
    ),
    GoniometerAxis(
        name=r"$\omega$",
        axis=sc.vector([0, 1, 0]),
        min=sc.scalar(-np.inf, unit="degree"),
        max=sc.scalar(np.inf, unit="degree"),
    ),
]
goni = Goniometer(axes)
_ = goni.plot()

In [ ]:
from IPython.display import display
f = plt.Figure()
plt._backend_mod.new_figure_manager_given_figure(1, f)
ax = f.add_subplot(projection='3d')
goni.plot(ax=ax)
f

In [ ]:
class Quantity(widgets.HBox):
    def __init__(self, *, units: Iterable[str], default_unit:str|None=None,
                description:str="", value:float=0.0, layout:Layout|None=None) -> None:
        number, unit = _make_quantity_inputs(
            units=units,
            default_unit=default_unit,
            description=description,
            value=value,
        )
        self._number = number
        self._unit = unit
        super().__init__((number, unit), layout=layout or Layout(display='flex', flex_flow='row', min_width='0'))

    @property
    def value(self) -> sc.Variable:
        return sc.scalar(self._number.value, unit=self._unit.value)

def _make_quantity_inputs(*, units: Iterable[str], default_unit:str|None,
                description:str, value:float)->tuple[widgets.FloatText, widgets.Dropdown]:
    units = list(units)
    if default_unit is None:
        default_unit = units[0]

    number = widgets.FloatText(
        value=value,
        description=description,
        layout=widgets.Layout(flex='1 1 auto', width='auto', min_width='0'),
        style={"description_width": "initial"},
    )

    # Width just wide enough for the longest option (plus room for the
    # dropdown arrow / padding). ~0.62em per character is a decent estimate
    # for typical UI fonts; add a fixed padding for the arrow/border.
    longest = max((len(u) for u in units), default=1)
    dropdown_width = f"{longest * 0.62 + 1.2}em"

    unit = widgets.Dropdown(
        options=units,
        value=default_unit,
        layout=widgets.Layout(width=dropdown_width, flex='0 0 auto'),
    )
    unit.add_class('sc-no-chevron')

    state = {"previous_unit": unit.value}
    def _on_unit_change(change):
        if change["name"] != "value":
            return
        old_unit = state["previous_unit"]
        new_unit = change["new"]
        state["previous_unit"] = new_unit

        try:
            number.value = sc.to_unit(sc.scalar(number.value, unit=old_unit), new_unit).value
        except Exception as exc:
            # Don't let a bad conversion crash the widget; surface the error.
            print(f"Unit conversion failed ({old_unit} -> {new_unit}): {exc}")

    unit.observe(_on_unit_change, names="value")

    return number, unit

In [ ]:
label_layout = Layout(flex='0 0 auto', width='auto', padding='0 0 0 2em')
flex_item_layout = Layout(flex='1 1 auto', width='auto', min_width='0')
button_layout = Layout(
    flex='0 0 auto',
    width='auto',
    height='13px',
    padding='0',
    margin='0',
    align_self='stretch',
)
_STYLE = """<style>
.sc-tiny-btn {
    display: flex !important;
    align-items: center;
    justify-content: center;
    line-height: 1 !important;
    padding: 0 !important;

    border-radius: 0 !important;
}
.sc-tiny-btn .fa, .sc-tiny-btn i {
    margin: 0 !important;
    font-size: 8px !important;
}
.sc-no-chevron select {
    appearance: none !important;
    -webkit-appearance: none !important;
    -moz-appearance: none !important;
    background-image: none !important;
    padding-right: 0.4em !important;   /* reclaim space the arrow used */
}
</style>"""


class GoniometerWidget(widgets.VBox):
    def __init__(self)->None:
        self._axes = GoniometerAxesWidget()
        super().__init__(
            (widgets.HTML(_STYLE), self._axes)
        )

    @property
    def value(self) -> Goniometer:
        return Goniometer(
            axes=self._axes.value
        )


class GoniometerAxesWidget(widgets.VBox):
    def __init__(self):
        self._add_axis_button = widgets.Button(description="Add axis", icon="plus")
        self._add_axis_button.on_click(lambda *args: self.add_axis())
        self._axes = []

        super().__init__((self._add_axis_button,), layout=Layout(width='100%'))

    @property
    def value(self) -> list[GoniometerAxis]:
        return [
            axis.value for axis in self._axes]

        
    def add_axis(self) -> None:
        self._axes.append(GoniometerAxisWidget(move_row=self._move_row))
        self._set_children()

    def _move_row(self, button, axis):
        idx = self._axes.index(axis)
        new_idx = idx + button._move
        if 0 <= new_idx < len(self._axes):
            self._axes[idx], self._axes[new_idx] = self._axes[new_idx], self._axes[idx]
            self._set_children()

    def _set_children(self):
        self.children = (*self._axes, self._add_axis_button)
        
            

class GoniometerAxisWidget(widgets.HBox):
    def __init__(self, *, move_row):
        children, inputs = _make_goniometer_axis_row(lambda button: move_row(button, self))
        super().__init__(children, layout=Layout(width='100%', display='flex', flex_flow='row', align_items='center'))
        
        self._inputs = inputs

    @property
    def value(self) -> GoniometerAxis:
        ax = self._inputs["axis"].value
        axis = sc.vector([
            1 * ("X" in ax),
            1 * ("Y" in ax),
            1 * ("Z" in ax),
        ]) * (-1 if "-" in ax else +1)
        return GoniometerAxis(
            axis=axis,
            name=self._inputs["name"].value,
            min=self._inputs["lo"].value,
            max=self._inputs["hi"].value,  # TODO rename to min, max
        )
    

def _make_goniometer_axis_row(move_row):
    inputs = {}
    
    inputs['name'] = name = widgets.Text(placeholder="Name", layout=flex_item_layout)
    axis_label = widgets.Label(value="Axis", layout=label_layout)
    inputs['axis'] = axis = widgets.Dropdown(
        options=("+X", "-X", "+Y", "-Y", "+Z", "-Z"),
        value="+Y",
        description="Axis",
        layout=widgets.Layout(flex='1 1 auto', width='auto', min_width='11em'),
    )
    limits_label = widgets.Label(value="Limits", layout=label_layout)
    inputs['lo'] = lo = Quantity(value=-np.inf, units=('deg', 'rad'), layout=flex_item_layout)
    inputs['hi'] = hi = Quantity(value=np.inf, units=('deg', 'rad'), layout=flex_item_layout)

    up_btn = widgets.Button(icon='caret-up', layout=button_layout)
    down_btn = widgets.Button(icon='caret-down', layout=button_layout)
    up_btn.add_class('sc-tiny-btn')
    down_btn.add_class('sc-tiny-btn')
    
    up_btn._move = -1
    down_btn._move = 1
    up_btn.on_click(move_row)
    down_btn.on_click(move_row)

    buttons = widgets.VBox((up_btn, down_btn),
                          layout=Layout(width='30px', height='28px', flex='0 0 auto', justify_content='space-between')
            )
    
    children =        (buttons, name, axis, limits_label, lo, hi)
    return children, inputs

    
goni = GoniometerWidget()
goni

In [ ]:
goni.value.axes